# The results

Everything here reads scores.csv, so a figure and a table cannot disagree. The
view is stated on every table, because dropping refusals changes what a score
means and the reader should know which they are looking at.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

In [ ]:
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
%load_ext autoreload
%autoreload 2

import backends
import evaluate
import prompts as templates
import run
import settings
import utils

utils.make_directories()
pd.set_option('display.max_colwidth', 90)
# Built from the dataset rather than shipped, so that what the pipeline reads is
# always derived from what is in data/halomi rather than from a stale copy.
if not settings.ITEMS_PATH.exists():
    raise SystemExit(
        'Nothing has been built yet. From the repository root, run:\n'
        '    python scripts/build.py\n'
        'That writes data/benchmark/items.csv and prompts.csv, which every '
        'notebook and stage reads.')

# and this notebook reads scores.csv, which only exists once replies have been
# collected and judged
if not settings.SCORES_PATH.exists():
    raise SystemExit(
        'Nothing has been scored yet. From the repository root, run:\n'
        '    python scripts/run.py generate --model <id>\n'
        '    python scripts/evaluate.py\n'
        'The model identifiers are in config/settings.yml under models.')

print('Ready')

## Headline

In [ ]:
scores = utils.read_table(settings.SCORES_PATH)
headline = scores[(scores['direction'] == 'overall') &
                  (scores['view'] == 'judged')]
display(headline.pivot_table(index=['method', 'shots'], columns='model',
                             values='mcc', aggfunc='first'))

## How much the view matters

If the three views differ substantially, refusals are doing real work and the
choice of view is a finding rather than a housekeeping decision.

In [ ]:
views = scores[(scores['direction'] == 'overall') &
               (scores['method'] == 'baseline')]
display(views.pivot_table(index='model', columns='view', values='mcc',
                          aggfunc='first')[['all', 'parsed', 'judged']])

## By direction

The question the multilingual framing exists to ask. A judge that is reliable on
Spanish and unreliable on Manipuri is not a reliable judge.

In [ ]:
per = scores[(scores['direction'] != 'overall') &
             (scores['view'] == 'judged') &
             (scores['method'] == 'baseline')]
display(per.pivot_table(index='direction', columns='model', values='mcc',
                        aggfunc='first').sort_values(
    per['model'].iloc[0] if len(per) else 'mcc'))

## Figures

In [ ]:
import figures

scores = utils.read_table(settings.SCORES_PATH)
for draw in [figures.figure_methods, figures.figure_directions,
             figures.figure_views, figures.figure_shots]:
    path = draw(scores)
    if path:
        print(path.name)